# Electricity Consumption Anomaly Detection
## Using Unsupervised Machine Learning

Flow: Dataset -> Preprocessing -> Feature Extraction -> K-Means -> PCA -> Isolation Forest -> Results

---

## STEP 1 - Dataset Loading and Exploration

What we are doing: Loading the LD2011_2014.txt file and exploring its structure.

Why: Before applying machine learning, we must understand what our data looks like.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported')

### About the Dataset

- File: LD2011_2014.txt from UCI Machine Learning Repository
- Contains electricity consumption (kW) of 370 Portuguese clients
- Readings every 15 minutes from January 2011 to December 2014
- File uses semicolon as separator and comma as decimal point (European format)
- Full file is 711 MB, so we load only 10000 rows and 50 clients


In [ ]:
print('Loading dataset, please wait...')

df = pd.read_csv(
    '../data/LD2011_2014.txt',
    sep=';',
    decimal=',',
    index_col=0,
    skiprows=lambda x: x > 0 and x < 100000,
    nrows=10000
)

df = df.iloc[:, :50]

print('Dataset loaded')
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

In [ ]:
# Show first 5 rows
df.head(5)

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print()
print('Column names:')
print(list(df.columns))

In [ ]:
# Convert timestamps from plain text to proper datetime format
df.index = pd.to_datetime(df.index)

print('Start date:', df.index[0])
print('End date  :', df.index[-1])
print('Timestamps converted to datetime')

In [ ]:
print('Data types:')
print(df.dtypes.value_counts())

In [ ]:
total_missing = df.isnull().sum().sum()
print('Total missing values:', total_missing)

cols_with_missing = df.isnull().sum()
cols_with_missing = cols_with_missing[cols_with_missing > 0]

if len(cols_with_missing) == 0:
    print('No missing values found')
else:
    print(cols_with_missing)

In [ ]:
print('Basic statistics for first 5 clients:')
df.iloc[:, :5].describe().round(4)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('Electricity Consumption - 3 Clients (First 10 Days)', fontsize=13)

clients = ['MT_001', 'MT_010', 'MT_025']
colors = ['steelblue', 'tomato', 'seagreen']

for i, (client, color) in enumerate(zip(clients, colors)):
    axes[i].plot(df[client][:1000].values, color=color, linewidth=0.9)
    axes[i].set_title(client)
    axes[i].set_xlabel('Time slot (every 15 minutes)')
    axes[i].set_ylabel('Consumption (kW)')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/step1_preview_plot.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
df.to_csv('../data/electricity_subset.csv')

print('Saved to: data/electricity_subset.csv')
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

---
## Step 1 Summary

| Item | Value |
|------|-------|
| Dataset File | LD2011_2014.txt |
| Rows Loaded | 10000 |
| Columns Used | 50 clients (MT_001 to MT_050) |
| Time Period | Jan 2011 to approx Apr 2011 |
| Reading Frequency | Every 15 minutes |

Run all cells above and confirm your output before moving to Step 2.

---
## STEP 2 - Data Preprocessing

What we are doing: Cleaning and preparing the data so machine learning algorithms can use it properly.

Why: Raw data is often messy. ML algorithms need clean, consistently scaled numbers to work correctly.

Steps in this section:
1. Load the saved subset
2. Handle missing values
3. Normalize (scale) the data
4. Confirm the data is ready

---

In [ ]:
# Load the subset we saved in Step 1
df = pd.read_csv('../data/electricity_subset.csv', index_col=0, parse_dates=True)

print('Loaded electricity_subset.csv')
print('Shape:', df.shape)

### 2.1 Handle Missing Values

Missing values mean some readings were not recorded.
We fill them using the average of the nearby values (forward fill then backward fill).
This is a simple and safe method for time series data.


In [ ]:
print('Missing values before:', df.isnull().sum().sum())

# ffill fills a missing value with the previous value
# bfill fills any remaining gaps with the next value
df = df.ffill().bfill()

print('Missing values after :', df.isnull().sum().sum())

### 2.2 Normalize the Data

Different clients use very different amounts of electricity.
For example, a factory may use 5000 kW while a house uses 50 kW.
If we do not normalize, the factory data will dominate the analysis and hide smaller patterns.

StandardScaler transforms every column so that:
- Mean becomes 0
- Standard deviation becomes 1

This puts all clients on the same scale so the algorithm treats them equally.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit_transform learns the mean and std of each column, then scales all values
df_scaled = scaler.fit_transform(df)

# Convert back to a DataFrame so we keep column names and index
df_scaled = pd.DataFrame(df_scaled, columns=df.columns, index=df.index)

print('Data normalized')
print('Shape:', df_scaled.shape)

### 2.3 Confirm the Normalized Data

After normalization, the mean of each column should be close to 0
and the standard deviation should be close to 1.
This confirms the scaling worked correctly.


In [ ]:
print('Statistics after normalization (first 5 clients):')
df_scaled.iloc[:, :5].describe().round(4)

In [ ]:
# Compare one client before and after normalization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('MT_001 - Before and After Normalization', fontsize=12)

axes[0].plot(df['MT_001'][:500].values, color='steelblue', linewidth=0.9)
axes[0].set_title('Before Normalization')
axes[0].set_xlabel('Time slot')
axes[0].set_ylabel('Consumption (kW)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_scaled['MT_001'][:500].values, color='tomato', linewidth=0.9)
axes[1].set_title('After Normalization')
axes[1].set_xlabel('Time slot')
axes[1].set_ylabel('Scaled value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/step2_normalization.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
df_scaled.to_csv('../data/electricity_scaled.csv')

print('Scaled data saved to: data/electricity_scaled.csv')
print('Shape:', df_scaled.shape)

---
## Step 2 Summary

| Task | Result |
|------|--------|
| Missing values | Filled using forward fill and backward fill |
| Normalization | StandardScaler applied (mean=0, std=1) |
| Saved file | data/electricity_scaled.csv |

The data is now clean and normalized. Ready for Step 3 - Feature Extraction.

---
## STEP 3 - Feature Extraction

What we are doing: Creating a small set of numbers that describe each client's electricity consumption behavior.

Why: Right now our dataset has 10000 rows per client. We cannot feed all of that directly into K-Means or Isolation Forest. Instead, we summarize each client into a few key numbers called features.

After this step we will have one row per client and a few feature columns.

Features we will create:
- mean: average consumption (typical usage level)
- max: highest consumption ever recorded (peak demand)
- min: lowest consumption recorded
- std: standard deviation (how much usage varies day to day)
- peak_to_avg: ratio of max to mean (detects extreme spikes)

---

In [ ]:
# Load the scaled data saved in Step 2
df_scaled = pd.read_csv('../data/electricity_scaled.csv', index_col=0, parse_dates=True)

print('Loaded electricity_scaled.csv')
print('Shape:', df_scaled.shape)

### 3.1 Build the Feature Table

We compute each feature across all 10000 time slots for every client.
The result is a table with 50 rows (one per client) and 5 columns (one per feature).


In [ ]:
features = pd.DataFrame()

# Each column in df_scaled is one client
# We summarize that client's 10000 readings into 5 numbers

features['mean'] = df_scaled.mean()         # average consumption
features['max']  = df_scaled.max()          # highest single reading
features['min']  = df_scaled.min()          # lowest single reading
features['std']  = df_scaled.std()          # how much usage varies

# peak_to_avg: how many times larger the peak is compared to the average
# A very high ratio can indicate abnormal spikes
# We add a small number (0.001) to avoid dividing by zero
features['peak_to_avg'] = features['max'] / (features['mean'].abs() + 0.001)

print('Feature table shape:', features.shape)
print('(', features.shape[0], 'clients x', features.shape[1], 'features )')

In [ ]:
print('First 10 rows of the feature table:')
features.head(10)

### 3.2 Why each feature helps detect anomalies

| Feature | What it measures | Why useful |
|---------|-----------------|------------|
| mean | Typical usage level | A very high or low mean is unusual |
| max | Peak demand | An abnormally high peak can indicate a fault or theft |
| min | Lowest reading | Consistently zero may mean meter error |
| std | Variation in usage | Very irregular usage patterns stand out |
| peak_to_avg | Spike intensity | A large spike relative to average is suspicious |


In [ ]:
print('Feature statistics:')
features.describe().round(4)

In [ ]:
# Bar chart showing mean consumption per client
plt.figure(figsize=(14, 4))
plt.bar(range(len(features)), features['mean'], color='steelblue')
plt.title('Average Consumption per Client (Normalized)')
plt.xlabel('Client index')
plt.ylabel('Mean scaled consumption')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../data/step3_mean_per_client.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
features.to_csv('../data/electricity_features.csv')

print('Features saved to: data/electricity_features.csv')
print('Shape:', features.shape)

---
## Step 3 Summary

| Feature | Description |
|---------|-------------|
| mean | Average consumption per client |
| max | Highest reading per client |
| min | Lowest reading per client |
| std | Variation in consumption per client |
| peak_to_avg | Ratio of peak to average (spike indicator) |

We now have a feature table with 50 rows and 5 columns.
This compact table is what K-Means, PCA, and Isolation Forest will work on.

Ready for Step 4 - K-Means Clustering.

---
## STEP 4 - K-Means Clustering

What we are doing: Grouping the 50 clients into clusters based on how similar their electricity consumption patterns are.

Why: Clients with similar usage behavior belong together. Once we know the normal groups, any client that does not fit well into any group becomes a candidate for an anomaly.

How K-Means works:
1. You choose the number of clusters K
2. The algorithm places K center points randomly
3. Each client is assigned to the nearest center
4. Centers are recalculated as the average of their group
5. Steps 3 and 4 repeat until the groups stop changing

---

In [ ]:
features = pd.read_csv('../data/electricity_features.csv', index_col=0)

print('Loaded electricity_features.csv')
print('Shape:', features.shape)

### 4.1 Choose the Number of Clusters (Elbow Method)

We do not know the best value of K in advance.
The Elbow Method tries different values of K and records the inertia (total distance of points from their cluster center).
When the inertia stops dropping sharply, that is the best K to use.
On the graph, look for the point where the line bends like an elbow.


In [ ]:
from sklearn.cluster import KMeans

inertia_values = []
k_range = range(2, 11)

for k in k_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(features)
    inertia_values.append(model.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertia_values, marker='o', color='steelblue')
plt.title('Elbow Method - Finding Best K')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia')
plt.xticks(list(k_range))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/step4_elbow.png', dpi=100, bbox_inches='tight')
plt.show()

print('Look at the graph above.')
print('The point where the line bends sharply is the best K.')

### 4.2 Apply K-Means with Chosen K

Based on the elbow graph, we pick K = 3.
This means we will group the 50 clients into 3 clusters:
- Low consumption clients
- Medium consumption clients
- High consumption clients

You can change K to 4 or 5 if your elbow graph shows a different bend.


In [ ]:
K = 3

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(features)

# Assign a cluster label (0, 1, or 2) to each client
features['cluster'] = kmeans.labels_

print('K-Means applied with K =', K)
print()
print('Cluster assignments:')
print(features['cluster'].value_counts().sort_index())

In [ ]:
print('Clients in each cluster:')
print()
for cluster_id in sorted(features['cluster'].unique()):
    clients_in_cluster = features[features['cluster'] == cluster_id].index.tolist()
    print(f'Cluster {cluster_id} ({len(clients_in_cluster)} clients):', clients_in_cluster)

In [ ]:
print('Average feature values per cluster:')
features.groupby('cluster')[['mean', 'max', 'min', 'std', 'peak_to_avg']].mean().round(4)

In [ ]:
# Bar chart showing how many clients are in each cluster
cluster_counts = features['cluster'].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(cluster_counts.index, cluster_counts.values, color=['steelblue', 'tomato', 'seagreen'])
plt.title('Number of Clients per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of clients')
plt.xticks([0, 1, 2], ['Cluster 0', 'Cluster 1', 'Cluster 2'])
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../data/step4_cluster_sizes.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
features.to_csv('../data/electricity_features.csv')

print('Features with cluster labels saved to: data/electricity_features.csv')

---
## Step 4 Summary

| Item | Value |
|------|-------|
| Algorithm | K-Means Clustering |
| Number of clusters (K) | 3 |
| Input | 50 clients x 5 features |
| Output | Each client assigned to Cluster 0, 1, or 2 |

What each cluster likely represents:
- Cluster with low mean and low std: low and steady consumers
- Cluster with medium mean: average consumers
- Cluster with high mean or high peak_to_avg: heavy or irregular consumers

The actual meaning depends on the values you see in the cluster summary table above.

Ready for Step 5 - PCA.

---
## STEP 5 - PCA (Principal Component Analysis)

What we are doing: Reducing the 5 features we created down to just 2 numbers so we can draw a 2D graph.

Why: We have 5 features per client. We cannot draw a 5-dimensional graph. PCA compresses those 5 features into 2 values (called principal components) while keeping as much information as possible.

Simple explanation of PCA:
Imagine you have a shadow of a 3D object on a wall. The shadow is 2D but still shows the overall shape.
PCA does the same thing with data. It finds the best angle to project the data so we lose as little information as possible.

After PCA we will have a 2D scatter plot where each dot is one client.
Dots that are close together have similar consumption behavior.
Dots that are far away from others may be anomalies.

---

In [ ]:
features = pd.read_csv('../data/electricity_features.csv', index_col=0)

print('Loaded electricity_features.csv')
print('Shape:', features.shape)

### 5.1 Apply PCA

We apply PCA on the 5 feature columns only (not the cluster column).
n_components=2 means we want 2 output values per client.


In [ ]:
from sklearn.decomposition import PCA

feature_cols = ['mean', 'max', 'min', 'std', 'peak_to_avg']

pca = PCA(n_components=2)
pca_result = pca.fit_transform(features[feature_cols])

# Store results in a DataFrame
df_pca = pd.DataFrame(pca_result, columns=['PC1', 'PC2'], index=features.index)
df_pca['cluster'] = features['cluster'].values

print('PCA applied')
print('Original features:', len(feature_cols))
print('After PCA:', df_pca.shape[1] - 1, 'components')

### 5.2 How Much Information Did We Keep?

explained_variance_ratio tells us what percentage of the original information is kept in each component.
If PC1 + PC2 together explain 80% or more, the 2D plot is a good representation of the data.


In [ ]:
explained = pca.explained_variance_ratio_ * 100

print('Variance explained by PC1:', round(explained[0], 2), '%')
print('Variance explained by PC2:', round(explained[1], 2), '%')
print('Total variance kept      :', round(explained.sum(), 2), '%')

### 5.3 PCA 2D Scatter Plot

Each dot in the graph below is one electricity client.
The colour shows which cluster the client belongs to (from Step 4).
Dots that are far from all others may be unusual clients.


In [ ]:
colors = {0: 'steelblue', 1: 'tomato', 2: 'seagreen'}

plt.figure(figsize=(9, 6))

for cluster_id in sorted(df_pca['cluster'].unique()):
    subset = df_pca[df_pca['cluster'] == cluster_id]
    plt.scatter(
        subset['PC1'],
        subset['PC2'],
        label='Cluster ' + str(cluster_id),
        color=colors[cluster_id],
        s=80,
        edgecolors='black',
        linewidths=0.5
    )

# Label each dot with the client name
for client in df_pca.index:
    plt.annotate(
        client,
        (df_pca.loc[client, 'PC1'], df_pca.loc[client, 'PC2']),
        fontsize=6,
        xytext=(3, 3),
        textcoords='offset points'
    )

plt.title('PCA - 2D View of All Clients (Coloured by Cluster)')
plt.xlabel('Principal Component 1 (' + str(round(explained[0], 1)) + '% variance)')
plt.ylabel('Principal Component 2 (' + str(round(explained[1], 1)) + '% variance)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/step5_pca_plot.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
df_pca.to_csv('../data/electricity_pca.csv')

print('PCA results saved to: data/electricity_pca.csv')
print('Shape:', df_pca.shape)

---
## Step 5 Summary

| Item | Value |
|------|-------|
| Algorithm | PCA (Principal Component Analysis) |
| Input | 50 clients x 5 features |
| Output | 50 clients x 2 components (PC1 and PC2) |
| Total variance kept | Check your output above |

What to look for in the PCA graph:
- Clients that are far away from all clusters are likely anomalies
- Clients inside tight groups are normal consumers
- Each cluster should appear as a visible group of dots

Ready for Step 6 - Isolation Forest.

---
## STEP 6 - Isolation Forest (Anomaly Detection)

What we are doing: Finding clients whose electricity usage is abnormal.

Why: Isolation Forest is designed exactly for this. Normal points are clustered together in dense areas. Anomalies are scattered far away. Isolation Forest isolates anomalies by randomly cutting the data until points are separated. Anomalies get separated very quickly because they are already isolated.

Key parameter - contamination:
This tells the algorithm what percentage of the data we expect to be anomalies.
We will set it to 10% (0.10). This means the algorithm will flag the 10% most unusual clients.

---

In [ ]:
features = pd.read_csv('../data/electricity_features.csv', index_col=0)
df_pca = pd.read_csv('../data/electricity_pca.csv', index_col=0)

print('Loaded features and PCA results')

### 6.1 Apply Isolation Forest

We apply the algorithm to the 5 main features.
The model returns:
- +1 for normal clients
- -1 for anomalies


In [ ]:
from sklearn.ensemble import IsolationForest

feature_cols = ['mean', 'max', 'min', 'std', 'peak_to_avg']

# Set contamination=0.10 to flag the 10% most unusual clients
iso_forest = IsolationForest(contamination=0.10, random_state=42)
iso_forest.fit(features[feature_cols])

# Predict anomalies (+1 = normal, -1 = anomaly)
predictions = iso_forest.predict(features[feature_cols])

# Convert predictions to a readable format
features['is_anomaly'] = ['Anomaly' if x == -1 else 'Normal' for x in predictions]

# 5. Anomaly scoring mechanism: Extract anomaly scores (lower scores = more anomalous)
features['anomaly_score'] = iso_forest.decision_function(features[feature_cols])

print('Isolation Forest applied and scores calculated!')

### 6.2 View the Results

Let us see how many normal and anomalous clients were found.


In [ ]:
print('Anomaly Counts:')
print(features['is_anomaly'].value_counts())

print('\nList of Anomalies:')
anomalies = features[features['is_anomaly'] == 'Anomaly'].index.tolist()
print(anomalies)

print('\nTop 5 Anomalies with their Scores (lowest score is most unusual):')
print(features.loc[anomalies, ['anomaly_score']].sort_values('anomaly_score').head(5))

### 6.3 What makes them anomalies?

Let us compare the average feature values of normal clients vs anomalies.


In [ ]:
features.groupby('is_anomaly')[feature_cols].mean().round(4)

### 6.4 Compare with Local Outlier Factor (LOF)

Local Outlier Factor (LOF) is another anomaly detection method. Instead of randomly splitting data, it looks at how crowded a client's neighborhood is.
Let us see if LOF agrees with Isolation Forest!


In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# Use the same 10% contamination
lof = LocalOutlierFactor(contamination=0.10)
lof_predictions = lof.fit_predict(features[feature_cols])

features['lof_anomaly'] = ['Anomaly' if x == -1 else 'Normal' for x in lof_predictions]
lof_anomalies = features[features['lof_anomaly'] == 'Anomaly'].index.tolist()

print('Clients flagged as Anomaly by LOF:')
print(lof_anomalies)

print('\nDo both models agree?')
agreed = set(anomalies).intersection(set(lof_anomalies))
print('Models both flagged:', list(agreed))


In [ ]:
# Add anomaly label to PCA dataframe for plotting in Step 7
df_pca['is_anomaly'] = features['is_anomaly'].values

features.to_csv('../data/electricity_features.csv')
df_pca.to_csv('../data/electricity_pca.csv')

print('Saved results to CSV files')

---
## Step 6 Summary

| Item | Value |
|------|-------|
| Algorithm | Isolation Forest |
| Contamination | 10% (0.10) |
| Normal Clients | Check output above |
| Anomalies Found | Check output above |

We successfully identified the anomalous clients. They usually have a much higher peak_to_avg ratio, or extremely high max consumption compared to normal clients.

Ready for Step 7 - Visualization.

---
## STEP 7 - Visualization

What we are doing: Creating simple graphs to visually see the anomalies.

Why: Numbers are hard to read. Graphs make it obvious why the algorithm flagged certain clients as anomalous.

Graphs we will create:
1. PCA Scatter Plot: Normal clients vs Anomalies
2. Time Series Plot: Comparing a normal client's raw usage to an anomalous client's usage

---

In [ ]:
df_pca = pd.read_csv('../data/electricity_pca.csv', index_col=0)
df_raw = pd.read_csv('../data/electricity_subset.csv', index_col=0, parse_dates=True)

print('Loaded data for visualization')

### 7.1 PCA Plot: Normal vs Anomaly

This is the same PCA plot from Step 5, but now we colour points by Normal (blue) or Anomaly (red).


In [ ]:
plt.figure(figsize=(9, 6))

# Plot normal clients
normal = df_pca[df_pca['is_anomaly'] == 'Normal']
plt.scatter(normal['PC1'], normal['PC2'], color='steelblue', label='Normal', s=60, edgecolors='black', linewidths=0.5)

# Plot anomalies
anomalies = df_pca[df_pca['is_anomaly'] == 'Anomaly']
plt.scatter(anomalies['PC1'], anomalies['PC2'], color='red', label='Anomaly', s=100, marker='X', edgecolors='black')

plt.title('Anomaly Detection Results (PCA 2D View)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/step7_anomaly_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

print('Notice how the red X marks (anomalies) are generally far away from the main blue clusters.')

### 7.2 Time Series: Normal vs Anomaly

Let us look at the actual electricity usage over time.
We pick one normal client and one anomalous client from our lists and compare their graphs.


In [ ]:
# Pick the first normal client and first anomaly
normal_client = normal.index[0]
anomaly_client = anomalies.index[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Raw Electricity Consumption: Normal vs Anomaly', fontsize=14, fontweight='bold')

# Plot normal
axes[0].plot(df_raw[normal_client][:1000].values, color='steelblue', linewidth=1)
axes[0].set_title(f'Normal Client: {normal_client}')
axes[0].set_ylabel('Consumption (kW)')
axes[0].grid(True, alpha=0.3)

# Plot anomaly
axes[1].plot(df_raw[anomaly_client][:1000].values, color='red', linewidth=1)
axes[1].set_title(f'Anomalous Client: {anomaly_client}')
axes[1].set_xlabel('Time slot')
axes[1].set_ylabel('Consumption (kW)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/step7_anomaly_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

print('Look at the Y-axis (Consumption values) and the shape of the spikes.')
print('Anomalies often have massive spikes or very strange usage patterns compared to normal clients.')

---
## STEP 8 - Final Results

Congratulations! You have completed the project.
Here is the final summary of what we achieved using unsupervised machine learning.


In [ ]:
print('================================================')
print('🌟 FINAL PROJECT RESULTS 🌟')
print('================================================')
print('Dataset Used     : UCI ElectricityLoadDiagrams')
print('Clients Analyzed :', len(df_pca))
print('Method Used      : Unsupervised Learning (no manual labels)')
print()
print('1. K-Means Clustering:')
print('   - Grouped clients into 3 distinct consumption profiles.')
print()
print('2. PCA:')
print('   - Reduced 5 features to 2 dimensions for visualization.')
print()
print('3. Isolation Forest:')
print('   - Successfully detected unusual consumption behavior.')
print(f'   - Normal Clients Found    : {len(normal)}')
print(f'   - Anomalous Clients Found : {len(anomalies)}')
print()
print('List of Anomalous Clients:')
print(list(anomalies.index))
print('================================================')